In [1]:
import pandas as pd

season = 2024

df = pd.read_parquet(fr'..\data\unprocessed\mens_sports_reference\sports_reference_{season}.parquet')

df

,Date,Team,Location,Opponent,Result,Team Score,Opponent Score,Team FG,Team FGA,Team FG%,...,Opponent TOV,Opponent PF,Overtimes Amount,Overtime,Score Differential,Adjusted Score Differential,Possessions,Team PPP,Opponent PPP,Tempo
0,2023-11-06,Abilene Christian,-1,Oklahoma State,1,64.0,59.0,26.0,57.0,0.456,...,12.0,16.0,0,0,5.0,1.253292,70.32,0.910125,0.839022,70.320000
1,2023-11-10,Abilene Christian,-1,NC State,-1,64.0,84.0,20.0,57.0,0.351,...,7.0,20.0,0,0,-20.0,1.500000,66.72,0.959233,1.258993,66.720000
2,2023-11-14,Abilene Christian,1,Prairie View,-1,74.0,79.0,23.0,60.0,0.383,...,14.0,23.0,0,0,-5.0,1.253292,71.86,1.029780,1.099360,71.860000
3,2023-11-17,Abilene Christian,0,San Jose State,1,77.0,71.0,25.0,59.0,0.424,...,6.0,18.0,0,0,6.0,1.285760,65.86,1.169147,1.078044,65.860000
4,2023-11-19,Abilene Christian,0,Fordham,1,59.0,45.0,22.0,55.0,0.400,...,19.0,16.0,0,0,14.0,1.448039,65.86,0.895840,0.683268,65.860000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11233,2024-02-17,Youngstown State,-1,Cleveland State,-1,73.0,81.0,25.0,58.0,0.431,...,9.0,22.0,0,0,-8.0,1.338710,72.46,1.007452,1.117858,72.460000
11234,2024-02-23,Youngstown State,-1,Milwaukee,1,84.0,80.0,32.0,78.0,0.410,...,14.0,23.0,1,1,4.0,1.214668,79.80,1.052632,1.002506,70.933333
11235,2024-02-25,Youngstown State,-1,Green Bay,1,71.0,59.0,26.0,55.0,0.473,...,10.0,15.0,0,0,12.0,1.417062,62.22,1.141112,0.948248,62.220000
11236,2024-02-28,Youngstown State,1,Detroit Mercy,1,69.0,55.0,26.0,61.0,0.426,...,9.0,14.0,0,0,14.0,1.448039,67.68,1.019504,0.812648,67.680000


Filter to just wins because I don't need duplicate perspectives

In [2]:
df = df.loc[
    df['Result'] == 1, 
    ['Date', 'Team', 'Location', 'Opponent', 'Result', 'Adjusted Score Differential']
].reset_index(drop=True)

df.sort_values(['Date'], inplace=True, ignore_index=True)

df

,Date,Team,Location,Opponent,Result,Adjusted Score Differential
0,2023-11-06,Abilene Christian,-1,Oklahoma State,1,1.253292
1,2023-11-06,Siena,1,Holy Cross,1,1.102120
2,2023-11-06,Seton Hall,1,Saint Peter's,1,1.399871
3,2023-11-06,Butler,1,Eastern Michigan,1,1.500000
4,2023-11-06,San Diego State,1,Cal State Fullerton,1,1.500000
...,...,...,...,...,...,...
5614,2024-03-19,Abilene Christian,0,Texas A&M-Corpus Christi,1,1.381279
5615,2024-03-19,Tarleton State,1,Texas Southern,1,1.399871
5616,2024-03-19,North Texas,-1,Louisiana State,1,1.313867
5617,2024-03-19,Colorado State,0,Virginia,1,1.500000


In [3]:
def rescale_weights(arr, minimum, maximum):
    """
    Rescale the weights array to match desired weights.
    Assumes the data is already transformed where a 1 point win is 1.00, a blowout is 1.50, and a tie is 0.50.
    Ties will be weighted as half the minimum.
    """

    arr_scaled = ((arr - 1.00) / 0.50) * (maximum - minimum) + minimum
    arr_scaled[arr == 0.50] = minimum / 2
    return arr_scaled

In [4]:
from typing import Tuple
from openskill.models import BradleyTerryFull
import numpy as np

def get_season_ratings(X_train_weights: np.array, sigma: float, hfa_mu: float) -> Tuple[BradleyTerryFull, dict]:
    """
    Run the algorithm over the course of a season.

    Args:
        X_train_weights (np.array): Numpy array where each row is a game and columns are Winner, Loser, Location, Weight.
        sigma (float): The starting variance for each team rating.
        hfa_mu (float): The mu of home field advantage.

    Returns:
        Tuple[BradleyTerryFull, dict]: The environment object and dictionary of team ratings.
    """
    # initialize
    env = BradleyTerryFull(sigma=sigma)

    teams = set(X_train_weights[:, 0]).union(set(X_train_weights[:, 1]))
    team_ratings = dict(zip(teams, [env.rating() for _ in range(len(teams))]))
    hfa = env.rating(mu=hfa_mu, sigma=0.0)

    # iterate through games
    for winner, loser, location, weight in X_train_weights:
        winner_rating = team_ratings[winner]
        loser_rating = team_ratings[loser]
        if location == 1:
            [[winner_post, _], [loser_post]] = env.rate([[winner_rating, hfa], [loser_rating]])
        elif location == -1:
            [[winner_post], [loser_post, _]] = env.rate([[winner_rating], [loser_rating, hfa]])
        else:
            [[winner_post], [loser_post]] = env.rate([[winner_rating], [loser_rating]])

        winner_mu_adjustment = (winner_post.mu - winner_rating.mu)*weight
        winner_sigma_adjustment = (winner_post.sigma - winner_rating.sigma)*weight
        team_ratings[winner] = env.rating(mu=winner_rating.mu + winner_mu_adjustment, sigma=winner_rating.sigma + winner_sigma_adjustment)

        loser_mu_adjustment = (loser_post.mu - loser_rating.mu)*weight
        loser_sigma_adjustment = (loser_post.sigma - loser_rating.sigma)*weight
        team_ratings[loser] = env.rating(mu=loser_rating.mu + loser_mu_adjustment, sigma=loser_rating.sigma + loser_sigma_adjustment)

    return env, team_ratings

In [5]:
hfa_mu = 0.50
sigma = 25/3
minimum = 2/3
maximum = 4/3


X_train = df[['Team', 'Opponent', 'Location']].to_numpy()
weights = rescale_weights(df['Adjusted Score Differential'].to_numpy(), minimum, maximum)

X_train_weights = np.column_stack((X_train, weights))

env, team_ratings = get_season_ratings(X_train_weights, sigma, hfa_mu)

len(team_ratings)

362

In [6]:
df_ratings = pd.DataFrame(
    {
        'Team': team_ratings.keys(),
        'Mu': [v.mu for v in team_ratings.values()],
        'Sigma': [v.sigma for v in team_ratings.values()],
    }
)

df_ratings['OS Rating'] = df_ratings['Mu'] - df_ratings['Sigma']*3

df_ratings.sort_values(['OS Rating', 'Mu'], ascending=False, ignore_index=True, inplace=True)

df_ratings.head(50)

,Team,Mu,Sigma,OS Rating
0,Connecticut,51.086743,4.163881,38.595099
1,Houston,50.517837,4.141423,38.093567
2,Iowa State,48.187216,3.951030,36.334127
3,Purdue,49.322721,4.367363,36.220631
4,Auburn,47.461159,3.908290,35.736290
5,North Carolina,46.051615,4.098798,33.755221
6,Illinois,44.993156,3.921646,33.228217
7,Tennessee,44.551923,4.089847,32.282383
8,South Carolina,43.598330,4.149501,31.149826
9,Nevada,43.346839,4.192624,30.768969


In [7]:
df_ratings.to_parquet(f'../data/preprocessed/mens_os_rankings/os_rankings_{season}.parquet')

'Done'

'Done'